# Parallel chains via Web Workers

`sampling()` runs one chain inside the notebook's own Pyodide kernel worker
— a single thread, like the rest of this project's notebooks. `StanModel.sampling_parallel(n_chains=...)`
instead spawns one plain-JS Web Worker per chain (no Pyodide in them, just
stanwasm's wasm module loaded fresh in each), so multiple chains can run at
once across whatever cores the browser gives them — something a single-threaded
Pyodide session can't do on its own.

This needs nested-Worker support (a Worker spawning another Worker): current
Chrome/Edge/Firefox, and Safari 16.4+ (March 2023). Where that's unavailable
it falls back to running the chains one at a time through `sampling()` and
prints a warning — it never just fails.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

import micropip
from pyodide.code import run_js

# self.location.origin alone is wrong on a GitHub-Pages-style project site
# (https://host/repo-name/) -- it omits "/repo-name". self.location.href
# resolves to the *kernel worker's own* script URL, which does include it,
# at a stable path under /extensions/ -- slice there to get the site root.
site_base = run_js("""
(() => {
  const href = self.location.href;
  const marker = "/extensions/";
  const idx = href.indexOf(marker);
  return idx === -1 ? self.location.origin : href.slice(0, idx);
})()
""")

wheel_url = site_base + "/files/pystanwasm-0.1.0-py3-none-any.whl"
await micropip.install(wheel_url)

import pystanwasm as stan
print("pystanwasm", stan.__version__)

## A model with real work per draw

10 predictors, 500 observations — small enough to run comfortably in a
browser tab, large enough (matrix products every gradient evaluation) that
a single chain takes a fraction of a second rather than a few milliseconds,
so parallelizing 4 of them is actually worth timing.

In [ ]:
rng = np.random.default_rng(7)
N, K = 500, 10
X = rng.normal(size=(N, K))
beta_true = rng.normal(0, 2, size=K)
alpha_true, sigma_true = 1.5, 1.2
y = alpha_true + X @ beta_true + rng.normal(0, sigma_true, size=N)

stan_code = """
data {
  int<lower=0> N;
  int<lower=0> K;
  matrix[N, K] X;
  vector[N] y;
}
parameters {
  real alpha;
  vector[K] beta;
  real<lower=0> sigma;
}
model {
  alpha ~ normal(0, 5);
  beta ~ normal(0, 5);
  sigma ~ exponential(1);
  y ~ normal(alpha + X * beta, sigma);
}
"""
data = {"N": N, "K": K, "X": X.tolist(), "y": y.tolist()}
N_CHAINS = 4

## Sequential: `N_CHAINS` calls to `sampling()`

In [ ]:
model = stan.StanModel(stan_code)

t0 = time.perf_counter()
sequential_fits = [
    await model.sampling(data=data, iter=2000, warmup=1000, seed=42 + i)
    for i in range(N_CHAINS)
]
sequential_time = time.perf_counter() - t0
print(f"sequential: {sequential_time:.2f}s for {N_CHAINS} chains")

## Parallel: one `sampling_parallel()` call

In [ ]:
t0 = time.perf_counter()
parallel_fits = await model.sampling_parallel(data=data, n_chains=N_CHAINS, iter=2000, warmup=1000)
parallel_time = time.perf_counter() - t0
print(f"parallel:   {parallel_time:.2f}s for {N_CHAINS} chains")
print(f"speedup:    {sequential_time / parallel_time:.1f}x")

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(["sequential", "parallel"], [sequential_time, parallel_time], color=["tab:gray", "tab:blue"])
plt.ylabel("wall-clock seconds")
plt.title(f"{N_CHAINS} chains, sequential vs Worker-parallel")
plt.show()

## Sanity check: same posterior either way

Parallelizing changes nothing about what's being computed — each chain is
independent either way, just with a different seed. `beta[0]`'s posterior
mean per chain should agree closely between the two runs.

In [ ]:
beta0_name = [n for n in sequential_fits[0].param_names if n.startswith("beta")][0]
seq_means = [f[beta0_name].mean() for f in sequential_fits]
par_means = [f[beta0_name].mean() for f in parallel_fits]
pd.DataFrame({"sequential": seq_means, "parallel": par_means}, index=[f"chain {i}" for i in range(N_CHAINS)])